# 04 — Stagnone di Marsala: 3D Hydrodynamic Model Setup

This notebook fixes the Model B configuration (`stagnone_py_lagoon3D`) to create a running 3D D-Flow FM model.

**Model B issues identified:**
1. AngLat=0, AngLon=0 → Coriolis OFF (should be ~37.86°N, 12.46°E)
2. InitialTemperature=6°C → Wrong for July Mediterranean (should be ~24°C)
3. InitialSalinity=30 ppt → Low for Mediterranean (should be ~37.5 ppt)
4. HisFile/MapFile blank → Should specify output names
5. CrsFile blank → No cross-sections for flux monitoring at inlets
6. Tlfsmo=0 → No boundary smoothing on startup
7. autoTimestep=1 (2D only) → Should be 3 or 5 for 3D
8. BackgroundTemperature=6°C → Should match initial
9. BackgroundSalinity=30 → Should match initial

**Approach:** Copy Model B input files to `model/dflowfm/`, fix the MDU using hydrolib-core, and prepare for execution.

## 1. Imports and paths

In [ ]:
%matplotlib inline
import os
import shutil
from pathlib import Path
import matplotlib.pyplot as plt

# Project paths
project_root = Path(r'F:\StagnoneDT')
model_b_dir = project_root / 'oldModel' / 'Stagnone_py_lagoon3D.dsproj_data' / 'Stagnone_dxy01_15m' / 'input'
model_a_dir = project_root / 'oldModel' / 'input'
output_dir = project_root / 'model' / 'dflowfm'

print(f'Model B input: {model_b_dir}')
print(f'Model A input: {model_a_dir}')
print(f'Output dir:    {output_dir}')
print(f'Model B exists: {model_b_dir.exists()}')

## 2. Copy Model B files to working model directory

We copy all Model B input files to `model/dflowfm/` so we can modify them without altering the reference.

In [ ]:
# Files to copy from Model B
files_to_copy = [
    # Grid and mesh
    'Stagnone_dxy01_15m_net.nc',
    # Boundary polylines
    'Stagnone_dxy01_15m.pli',
    'Mozia.pli',
    # Boundary condition files
    'waterlevelbnd_constant_Stagnone_dxy01_15m.bc',
    'waterlevelbnd_CMEMS_Stagnone_dxy01_15m.bc',
    'salinitybnd_CMEMS_Stagnone_dxy01_15m.bc',
    'temperaturebnd_CMEMS_Stagnone_dxy01_15m.bc',
    'tide_tpxo80_opendap_Stagnone_dxy01_15m.bc',
    'uxuyadvectionvelocitybnd_CMEMS_Stagnone_dxy01_15m.bc',
    'Tracer1.bc',
    # Initial conditions
    'initialtracerTracer1.xyz',
    # ERA5 meteorological forcing
    'era5_u10n_20250701to20250710_ERA5.nc',
    'era5_v10n_20250701to20250710_ERA5.nc',
    'era5_msl_20250701to20250710_ERA5.nc',
    'era5_chnk_20250701to20250710_ERA5.nc',
    # Nudging
    'nudge_salinity_temperature_2025-07-01_00-00-00.nc',
    # Observation and dry cells
    'Stagnone_dxy01_15m_obs.xyn',
    'illegalcells_dry.pol',
    # Land boundaries
    'sicily2.ldb',
    'Stagnone_dxy01_15m.ldb',
    # Initial fields
    'initialFields.ini',
]

os.makedirs(output_dir, exist_ok=True)

copied = []
missing = []
for f in files_to_copy:
    src = model_b_dir / f
    dst = output_dir / f
    if src.exists():
        shutil.copy2(src, dst)
        copied.append(f)
    else:
        missing.append(f)

print(f'Copied {len(copied)} files')
if missing:
    print(f'MISSING: {missing}')

## 3. Inspect the existing mesh

In [ ]:
import xugrid as xu
import contextily as ctx
import dfm_tools as dfmt

# Load and inspect the mesh
netfile = output_dir / 'Stagnone_dxy01_15m_net.nc'
uds = xu.open_dataset(str(netfile))
print(uds)
print(f'\nGrid info:')
print(f'  Nodes: {uds.grid.node_coordinates[0].size}')
print(f'  Faces: {uds.grid.face_coordinates[0].size}')

# Check bathymetry range
if 'mesh2d_node_z' in uds:
    z = uds['mesh2d_node_z']
    print(f'  Bathymetry range: {float(z.min()):.2f} to {float(z.max()):.2f} m')
elif 'mesh2d_face_z' in uds:
    z = uds['mesh2d_face_z']
    print(f'  Bathymetry range: {float(z.min()):.2f} to {float(z.max()):.2f} m')

In [ ]:
# Plot mesh with bathymetry
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Full domain
ax = axes[0]
ax.set_title('Full domain — mesh + bathymetry')
if 'mesh2d_node_z' in uds:
    uds.mesh2d_node_z.ugrid.plot(ax=ax, center=False, cmap='terrain', vmin=-50, vmax=5)
uds.grid.plot(ax=ax, linewidth=0.3, color='white', alpha=0.2)
dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')

# Lagoon zoom
ax = axes[1]
ax.set_title('Lagoon zoom')
if 'mesh2d_node_z' in uds:
    uds.mesh2d_node_z.ugrid.plot(ax=ax, center=False, cmap='terrain', vmin=-5, vmax=2)
uds.grid.plot(ax=ax, linewidth=0.3, color='white', alpha=0.2)
dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')
ax.set_xlim(12.41, 12.49)
ax.set_ylim(37.83, 37.92)

fig.tight_layout()
plt.savefig(str(project_root / 'figures' / 'mesh_bathymetry.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Fix and create corrected MDU using hydrolib-core

We load Model B's MDU, fix the identified issues, and save a corrected version.

In [ ]:
import hydrolib.core.dflowfm as hcdfm
import pandas as pd

# hydrolib-core 1.0.0 does not recognize some keywords added by DeltaShell GUI.
# We need to strip them from the MDU before loading.
mdu_original = model_b_dir / 'Stagnone_dxy01_15m.mdu'
mdu_cleaned = output_dir / 'Stagnone_dxy01_15m.mdu'

# Read MDU, remove unsupported keywords, write cleaned version
unknown_keywords = ['Umodlin', 'EffectSpiral', 'WaveNikuradse', 'S1incinterval']
with open(mdu_original, 'r') as f:
    lines = f.readlines()

with open(mdu_cleaned, 'w') as f:
    for line in lines:
        key = line.strip().split('=')[0].strip()
        if key not in unknown_keywords:
            f.write(line)
        else:
            print(f'  Removed unsupported keyword: {key}')

# Now load the cleaned MDU
mdu = hcdfm.FMModel(filepath=mdu_cleaned)

print('\nLoaded Model B MDU successfully')
print(f'  Net file: {mdu.geometry.netfile}')
print(f'  Kmx (layers): {mdu.geometry.kmx}')
print(f'  AngLat: {mdu.geometry.anglat}')
print(f'  AngLon: {mdu.geometry.anglon}')
print(f'  Salinity: {mdu.physics.salinity}')
print(f'  Temperature: {mdu.physics.temperature}')
print(f'  InitialTemp: {mdu.physics.initialtemperature}')
print(f'  InitialSal: {mdu.physics.initialsalinity}')

In [ ]:
# ============================================================
# FIX 1: Enable Coriolis (CRITICAL for 3D circulation)
# ============================================================
# Stagnone center: 37.86°N, 12.46°E
mdu.geometry.anglat = 37.86
mdu.geometry.anglon = 12.46
print(f'FIX 1: Coriolis enabled — AngLat={mdu.geometry.anglat}, AngLon={mdu.geometry.anglon}')

# ============================================================
# FIX 2: Correct initial temperature for July Mediterranean
# ============================================================
mdu.physics.initialtemperature = 24.0  # was 6°C
mdu.physics.backgroundwatertemperature = 24.0  # was 6°C
print(f'FIX 2: InitialTemperature={mdu.physics.initialtemperature}°C (was 6°C)')

# ============================================================
# FIX 3: Correct initial salinity for Mediterranean
# ============================================================
mdu.physics.initialsalinity = 37.5  # was 30
mdu.physics.backgroundsalinity = 37.5  # was 30
print(f'FIX 3: InitialSalinity={mdu.physics.initialsalinity} ppt (was 30)')

# ============================================================
# FIX 4: Boundary smoothing to avoid abrupt start
# ============================================================
mdu.numerics.tlfsmo = 3600.0  # 1 hour Fourier smoothing on WL boundaries
print(f'FIX 4: Tlfsmo={mdu.numerics.tlfsmo}s (was 0)')

# ============================================================
# FIX 5: Auto timestep for 3D model
# ============================================================
mdu.time.autotimestep = 3  # 3D (hor. out) — was 1 (2D only)
print(f'FIX 5: autoTimestep={mdu.time.autotimestep} (was 1, 2D only)')

# ============================================================
# FIX 6: Increase background vertical viscosity slightly
# ============================================================
# k-epsilon handles turbulent mixing. Background values are molecular/minimum.
# 5e-5 may cause issues in very thin sigma layers. Increase slightly.
mdu.physics.vicoww = 1e-4  # was 5e-5
mdu.physics.dicoww = 1e-4  # was 5e-5
print(f'FIX 6: Vicoww={mdu.physics.vicoww}, Dicoww={mdu.physics.dicoww} (were 5e-5)')

print('\nAll fixes applied.')

In [ ]:
# ============================================================
# Update file references to point to working directory
# ============================================================
# The ext files don't exist yet in the working directory — they will be
# recreated in the next cells. For now, just set them to None to avoid
# file-not-found errors, then reassign after saving.

mdu.external_forcing.extforcefile = None
mdu.external_forcing.extforcefilenew = None

# Output directory
mdu.output.outputdir = 'output'

print('File references updated.')
print(f'  NetFile:       {mdu.geometry.netfile}')
print(f'  IniFieldFile:  {mdu.geometry.inifieldfile}')
print(f'  ObsFile:       {mdu.output.obsfile}')
print(f'  ExtForceFile:  will be recreated in next cells')

In [ ]:
# ============================================================
# Write corrected ext files
# ============================================================
# We recreate the ext files to ensure they reference local paths

from hydrolib.core.dflowfm.ext.models import MeteoForcingFileType

# --- New format ext (boundary conditions) ---
ext_new = hcdfm.ExtModel()

pli_file = 'Stagnone_dxy01_15m.pli'

# Water level: constant offset + tidal + CMEMS (additive)
ext_new.boundary.append(hcdfm.Boundary(
    quantity='waterlevelbnd',
    locationfile=pli_file,
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'waterlevelbnd_constant_Stagnone_dxy01_15m.bc')),
))
ext_new.boundary.append(hcdfm.Boundary(
    quantity='waterlevelbnd',
    locationfile=pli_file,
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'tide_tpxo80_opendap_Stagnone_dxy01_15m.bc')),
))
ext_new.boundary.append(hcdfm.Boundary(
    quantity='waterlevelbnd',
    locationfile=pli_file,
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'waterlevelbnd_CMEMS_Stagnone_dxy01_15m.bc')),
))

# Salinity boundary
ext_new.boundary.append(hcdfm.Boundary(
    quantity='salinitybnd',
    locationfile=pli_file,
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'salinitybnd_CMEMS_Stagnone_dxy01_15m.bc')),
))

# Temperature boundary
ext_new.boundary.append(hcdfm.Boundary(
    quantity='temperaturebnd',
    locationfile=pli_file,
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'temperaturebnd_CMEMS_Stagnone_dxy01_15m.bc')),
))

# Tracer at Mozia
ext_new.boundary.append(hcdfm.Boundary(
    quantity='tracerbndTracer1',
    locationfile='Mozia.pli',
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'Tracer1.bc')),
))

# Velocity boundary
ext_new.boundary.append(hcdfm.Boundary(
    quantity='uxuyadvectionvelocitybnd',
    locationfile=pli_file,
    forcingfile=hcdfm.ForcingModel(filepath=str(output_dir / 'uxuyadvectionvelocitybnd_CMEMS_Stagnone_dxy01_15m.bc')),
))

# Meteo: wind x
ext_new.meteo.append(hcdfm.Meteo(
    quantity='windx',
    forcingFile='era5_u10n_20250701to20250710_ERA5.nc',
    forcingFileType=MeteoForcingFileType.netcdf,
    forcingVariableName='u10n',
    interpolationMethod='linearSpaceTime',
    operand='O',
))

# Meteo: wind y
ext_new.meteo.append(hcdfm.Meteo(
    quantity='windy',
    forcingFile='era5_v10n_20250701to20250710_ERA5.nc',
    forcingFileType=MeteoForcingFileType.netcdf,
    forcingVariableName='v10n',
    interpolationMethod='linearSpaceTime',
    operand='O',
))

# Meteo: air pressure
ext_new.meteo.append(hcdfm.Meteo(
    quantity='airpressure',
    forcingFile='era5_msl_20250701to20250710_ERA5.nc',
    forcingFileType=MeteoForcingFileType.netcdf,
    forcingVariableName='msl',
    interpolationMethod='linearSpaceTime',
    operand='O',
))

# Meteo: Charnock coefficient
ext_new.meteo.append(hcdfm.Meteo(
    quantity='charnock',
    forcingFile='era5_chnk_20250701to20250710_ERA5.nc',
    forcingFileType=MeteoForcingFileType.netcdf,
    forcingVariableName='chnk',
    interpolationMethod='linearSpaceTime',
    operand='O',
))

print(f'New ext: {len(ext_new.boundary)} boundaries, {len(ext_new.meteo)} meteo entries')

In [ ]:
# --- Old format ext (nudging) ---
ext_old = hcdfm.ExtOldModel()

ext_old.forcing.append(hcdfm.ExtOldForcing(
    quantity='nudge_salinity_temperature',
    filename='nudge_salinity_temperature_2025-07-01_00-00-00.nc',
    filetype=11,  # ncgrid
    method=3,     # spatial-temporal interpolation with weight factors
    operand='O',  # override
))

print(f'Old ext: {len(ext_old.forcing)} forcing entries')

In [ ]:
# ============================================================
# Save corrected ext files and MDU
# ============================================================

# Save ext files first
ext_new_file = output_dir / 'Stagnone_dxy01_15m_new.ext'
ext_new.save(filepath=str(ext_new_file))
print(f'Saved: {ext_new_file}')

ext_old_file = output_dir / 'Stagnone_dxy01_15m_old.ext'
ext_old.save(filepath=str(ext_old_file))
print(f'Saved: {ext_old_file}')

# Now reassign ext files to the MDU (files exist on disk now)
mdu.external_forcing.extforcefile = ext_old
mdu.external_forcing.extforcefilenew = ext_new

# Save MDU
mdu_file = output_dir / 'Stagnone_dxy01_15m.mdu'
mdu.filepath = mdu_file
mdu.save(recurse=False)
print(f'Saved: {mdu_file}')

# Make paths relative for portability
try:
    dfmt.make_paths_relative(str(mdu_file))
    print('Paths made relative successfully')
except Exception as e:
    print(f'Note: make_paths_relative: {e}')

## 5. Verify the corrected model configuration

In [ ]:
# Reload and verify — strip unsupported keywords again since mdu.save() may reintroduce them
unknown_keywords = ['Umodlin', 'EffectSpiral', 'WaveNikuradse', 'S1incinterval']
with open(mdu_file, 'r') as f:
    lines = f.readlines()
with open(mdu_file, 'w') as f:
    for line in lines:
        key = line.strip().split('=')[0].strip()
        if key not in unknown_keywords:
            f.write(line)

mdu_check = hcdfm.FMModel(filepath=mdu_file)

print('=== CORRECTED MODEL CONFIGURATION ===')
print(f'\n[Geometry]')
print(f'  NetFile:    {mdu_check.geometry.netfile}')
print(f'  Kmx:        {mdu_check.geometry.kmx} vertical layers')
print(f'  LayerType:  {mdu_check.geometry.layertype} (1=sigma)')
print(f'  AngLat:     {mdu_check.geometry.anglat} (Coriolis latitude)')
print(f'  AngLon:     {mdu_check.geometry.anglon}')
print(f'  BedlevType: {mdu_check.geometry.bedlevtype}')

print(f'\n[Physics]')
print(f'  Friction:   Manning n={mdu_check.physics.uniffrictcoef}')
print(f'  Salinity:   {mdu_check.physics.salinity} (init={mdu_check.physics.initialsalinity} ppt)')
print(f'  Temperature: {mdu_check.physics.temperature} (init={mdu_check.physics.initialtemperature}°C)')
print(f'  Turbulence: {mdu_check.numerics.turbulencemodel} (3=k-epsilon)')
print(f'  Vicouv:     {mdu_check.physics.vicouv}')
print(f'  Vicoww:     {mdu_check.physics.vicoww}')
print(f'  Smagorinsky: {mdu_check.physics.smagorinsky}')
print(f'  Idensform:  {mdu_check.physics.idensform} (2=UNESCO)')

print(f'\n[Numerics]')
print(f'  CFLMax:     {mdu_check.numerics.cflmax}')
print(f'  Epshu:      {mdu_check.numerics.epshu}')
print(f'  Tlfsmo:     {mdu_check.numerics.tlfsmo}s')

print(f'\n[Wind]')
print(f'  ICdtyp:     {mdu_check.wind.icdtyp} (4=Charnock)')
print(f'  PavBnd:     {mdu_check.wind.pavbnd} Pa')

print(f'\n[Time]')
print(f'  RefDate:    {mdu_check.time.refdate}')
print(f'  Start:      {mdu_check.time.startdatetime}')
print(f'  Stop:       {mdu_check.time.stopdatetime}')
print(f'  DtMax:      {mdu_check.time.dtmax}s')
print(f'  DtInit:     {mdu_check.time.dtinit}s')
print(f'  AutoTimestep: {mdu_check.time.autotimestep}')

print(f'\n[Output]')
print(f'  HisInterval: {mdu_check.output.hisinterval}')
print(f'  MapInterval: {mdu_check.output.mapinterval}')
print(f'  ObsFile:     {mdu_check.output.obsfile}')

## 6. Verify boundary condition data

In [ ]:
import xarray as xr

# Check CMEMS water level boundary
bc_wl = hcdfm.ForcingModel(filepath=str(output_dir / 'waterlevelbnd_CMEMS_Stagnone_dxy01_15m.bc'))
print(f'Water level BC: {len(bc_wl.forcing)} forcing entries')
if bc_wl.forcing:
    f0 = bc_wl.forcing[0]
    print(f'  First entry: {f0.name}')
    ds = dfmt.forcinglike_to_Dataset(f0, convertnan=True)
    print(f'  Time range: {ds.time.values[0]} to {ds.time.values[-1]}')
    print(f'  WL range: {float(ds.waterlevelbnd.min()):.3f} to {float(ds.waterlevelbnd.max()):.3f} m')

# Check ERA5 wind
ds_wind_u = xr.open_dataset(str(output_dir / 'era5_u10n_20250701to20250710_ERA5.nc'))
ds_wind_v = xr.open_dataset(str(output_dir / 'era5_v10n_20250701to20250710_ERA5.nc'))
print(f'\nERA5 wind U: time {ds_wind_u.time.values[0]} to {ds_wind_u.time.values[-1]}')
print(f'ERA5 wind V: time {ds_wind_v.time.values[0]} to {ds_wind_v.time.values[-1]}')
print(f'Wind U range: {float(ds_wind_u.u10n.min()):.1f} to {float(ds_wind_u.u10n.max()):.1f} m/s')
print(f'Wind V range: {float(ds_wind_v.v10n.min()):.1f} to {float(ds_wind_v.v10n.max()):.1f} m/s')

In [ ]:
# Plot ERA5 wind for the simulation period
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ds_wind_u.u10n.isel(time=0).plot(ax=ax, cmap='RdBu_r')
ax.set_title('ERA5 u10n (first timestep)')
dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')

ax = axes[1]
ds_wind_v.v10n.isel(time=0).plot(ax=ax, cmap='RdBu_r')
ax.set_title('ERA5 v10n (first timestep)')
dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')

fig.tight_layout()
plt.savefig(str(project_root / 'figures' / 'era5_wind.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Generate DIMR execution files

In [ ]:
# Generate DIMR config and execution scripts
# Set dimrset_folder to your local Delft3D FM installation path, or None if not available
dimrset_folder = None  # e.g., r'C:\Program Files\Deltares\Delft3D FM Suite 2026.01 HMWQ\plugins\DeltaShell.Dimr\kernels'

nproc = 1  # number of parallel processes
dfmt.create_model_exec_files(
    file_mdu=str(mdu_file),
    nproc=nproc,
    dimrset_folder=dimrset_folder
)

print('Execution files generated.')
print(f'Model directory: {output_dir}')
print('\nFiles in model directory:')
for f in sorted(output_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:60s} {size_kb:8.1f} KB')

## 8. Summary of changes

| Parameter | Model B (original) | Corrected | Rationale |
|-----------|--------------------|-----------|-----------|
| AngLat | 0 | 37.86 | Enable Coriolis for 3D circulation |
| AngLon | 0 | 12.46 | Solar heat flux computation |
| InitialTemperature | 6°C | 24°C | July Mediterranean SST |
| BackgroundTemp | 6°C | 24°C | Consistent with initial |
| InitialSalinity | 30 ppt | 37.5 ppt | Mediterranean salinity |
| BackgroundSalinity | 30 ppt | 37.5 ppt | Consistent with initial |
| Vicoww | 5e-5 | 1e-4 | Slightly increased background vertical viscosity |
| Dicoww | 5e-5 | 1e-4 | Slightly increased background vertical diffusivity |
| Tlfsmo | 0 | 3600 | 1-hour boundary smoothing at startup |
| autoTimestep | 1 (2D) | 3 (3D hor) | Proper 3D auto-timestep |

### Next steps
1. Run the model locally or on EDITO for the 9-day test period (July 1-10, 2025)
2. Check output: water levels at BocaNord/BocaSud, vertical velocity profiles
3. Compare with in-situ station data (notebook 02_input_insitu_wl_wind)
4. Evaluate if domain needs northward extension for turbidity source 2 (notebook 00_setup_domain_mesh)
5. Add satellite-derived roughness (notebook 01_input_satellite_roughness)
6. Add SWAN wave coupling (notebook 13_build_v03_wave_coupling)